In [1]:
import pandas as pd
import numpy as np
import random
import pyspark.pandas as ps
from pyspark.sql import SparkSession

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
5,None,pyspark,idle,,,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
existing_spark = SparkSession.getActiveSession()

if existing_spark:
    print("Existing Spark session found. Stopping it...")
    existing_spark.stop()

In [ ]:
spark = (
        SparkSession.builder
        .appName("DataFrames App")
        .master("local[*]")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,"
                                        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
                                        "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0,"
                                        "org.apache.hadoop:hadoop-aws:3.3.4,"
                                        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
                                        "software.amazon.awssdk:glue:2.25.26,"
                                        "software.amazon.awssdk:sts:2.25.26,"
                                        "software.amazon.awssdk:s3:2.25.26,"
                                        "software.amazon.awssdk:dynamodb:2.25.26,"
                                        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
                                        "software.amazon.awssdk:url-connection-client:2.25.26")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,"
                                        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
                                        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
        .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.warehouse.dir", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.catalog.glue_catalog.client.region", "ap-south-1")
        .config("spark.sql.defaultCatalog", "spark_catalog")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("hive.metastore.client.factory.class", "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
        .enableHiveSupport()
        .getOrCreate()
    )

In [ ]:
spark

In [2]:
data = [(i, "user_" + str(i), random.randint(18, 60)) for i in range(30)]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
user = spark.sparkContext.parallelize(data)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
user.collect()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

[(0, 'user_0', 25), (1, 'user_1', 56), (2, 'user_2', 60), (3, 'user_3', 58), (4, 'user_4', 20), (5, 'user_5', 23), (6, 'user_6', 41), (7, 'user_7', 41), (8, 'user_8', 60), (9, 'user_9', 59), (10, 'user_10', 37), (11, 'user_11', 24), (12, 'user_12', 46), (13, 'user_13', 29), (14, 'user_14', 28), (15, 'user_15', 31), (16, 'user_16', 54), (17, 'user_17', 27), (18, 'user_18', 51), (19, 'user_19', 25), (20, 'user_20', 31), (21, 'user_21', 35), (22, 'user_22', 25), (23, 'user_23', 50), (24, 'user_24', 19), (25, 'user_25', 50), (26, 'user_26', 26), (27, 'user_27', 25), (28, 'user_28', 46), (29, 'user_29', 18)]

In [5]:
user_df = user.toDF()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
user_df.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---+-------+---+
| _1|     _2| _3|
+---+-------+---+
|  0| user_0| 25|
|  1| user_1| 56|
|  2| user_2| 60|
|  3| user_3| 58|
|  4| user_4| 20|
|  5| user_5| 23|
|  6| user_6| 41|
|  7| user_7| 41|
|  8| user_8| 60|
|  9| user_9| 59|
| 10|user_10| 37|
| 11|user_11| 24|
| 12|user_12| 46|
| 13|user_13| 29|
| 14|user_14| 28|
| 15|user_15| 31|
| 16|user_16| 54|
| 17|user_17| 27|
| 18|user_18| 51|
| 19|user_19| 25|
+---+-------+---+
only showing top 20 rows

In [7]:
df_user = user.toDF(['id', 'username', 'age'])

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [8]:
df_user.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---+--------+---+
| id|username|age|
+---+--------+---+
|  0|  user_0| 25|
|  1|  user_1| 56|
|  2|  user_2| 60|
|  3|  user_3| 58|
|  4|  user_4| 20|
|  5|  user_5| 23|
|  6|  user_6| 41|
|  7|  user_7| 41|
|  8|  user_8| 60|
|  9|  user_9| 59|
| 10| user_10| 37|
| 11| user_11| 24|
| 12| user_12| 46|
| 13| user_13| 29|
| 14| user_14| 28|
| 15| user_15| 31|
| 16| user_16| 54|
| 17| user_17| 27|
| 18| user_18| 51|
| 19| user_19| 25|
+---+--------+---+
only showing top 20 rows

In [9]:
cols = ("ID", "Username", "Age")
df_users = df_user.toDF(*cols)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
df_users.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---+--------+---+
| ID|Username|Age|
+---+--------+---+
|  0|  user_0| 25|
|  1|  user_1| 56|
|  2|  user_2| 60|
|  3|  user_3| 58|
|  4|  user_4| 20|
+---+--------+---+
only showing top 5 rows

In [11]:
df_user.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---+--------+---+
| id|username|age|
+---+--------+---+
|  0|  user_0| 25|
|  1|  user_1| 56|
|  2|  user_2| 60|
|  3|  user_3| 58|
|  4|  user_4| 20|
+---+--------+---+
only showing top 5 rows

In [12]:
pdf = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Cathy"],
    "age": [25, 30, 28]
})

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
pdf_df = spark.createDataFrame(pdf)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

/home/glue_user/spark/python/pyspark/sql/pandas/conversion.py:474: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():
/home/glue_user/spark/python/pyspark/sql/pandas/conversion.py:486: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():

In [14]:
pdf_df.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 25|
|  2|  Bob| 30|
|  3|Cathy| 28|
+---+-----+---+

In [15]:
pd_pdf_df = pdf_df.toPandas()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [16]:
pd_pdf_df.head()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

   id   name  age
0   1  Alice   25
1   2    Bob   30
2   3  Cathy   28

In [17]:
df = ps.from_pandas(pd_pdf_df)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

/home/glue_user/spark/python/pyspark/pandas/internal.py:1573: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  fields = [
/home/glue_user/spark/python/pyspark/sql/pandas/conversion.py:486: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  for column, series in pdf.iteritems():

In [18]:
df.head()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

   id   name  age
0   1  Alice   25
1   2    Bob   30
2   3  Cathy   28